In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
from torch.utils.data import DataLoader, Subset, random_split
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os
import json
import copy
from PIL import Image
import sys

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

os.makedirs("artifacts/figures", exist_ok=True)

BATCH_SIZE = 64
NUM_CLASSES_A = 10
NUM_EPOCHS = 15


Using device: cpu


In [3]:
transform_base = transforms.Compose([
    transforms.Resize((96, 96)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

transform_aug = transforms.Compose([
    transforms.Resize((96, 96)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

transform_resnet = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


In [4]:
train_dataset_full = torchvision.datasets.STL10(root='./data', split='train', download=True, transform=transform_base)
test_dataset = torchvision.datasets.STL10(root='./data', split='test', download=True, transform=transform_base)


100%|██████████████████████████████████████| 2.64G/2.64G [31:32<00:00, 1.40MB/s]


In [5]:
train_size = int(0.8 * len(train_dataset_full))
val_size = len(train_dataset_full) - train_size

train_dataset, val_dataset = random_split(train_dataset_full, [train_size, val_size], generator=torch.Generator().manual_seed(SEED))

train_loader_aug = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, 
                               collate_fn=lambda batch: torchvision.datasets.STL10(root='./data', split='train', download=True, transform=transform_aug).__getitem__(0))

train_dataset_aug = torchvision.datasets.STL10(root='./data', split='train', download=True, transform=transform_aug)
train_size_aug = int(0.8 * len(train_dataset_aug))
val_size_aug = len(train_dataset_aug) - train_size_aug
train_dataset_aug, val_dataset_aug = random_split(train_dataset_aug, [train_size_aug, val_size_aug], generator=torch.Generator().manual_seed(SEED))

train_loader_base = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader_base = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
train_loader_aug = DataLoader(train_dataset_aug, batch_size=BATCH_SIZE, shuffle=True)
val_loader_aug = DataLoader(val_dataset_aug, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Train size: {len(train_dataset)}, Val size: {len(val_dataset)}, Test size: {len(test_dataset)}")
x_batch, y_batch = next(iter(train_loader_base))
print(f"Batch shape: {x_batch.shape}, Labels shape: {y_batch.shape}")
print(f"Value range: [{x_batch.min():.2f}, {x_batch.max():.2f}]")
print(f"Number of classes: {len(torch.unique(y_batch))}")


Train size: 4000, Val size: 1000, Test size: 8000
Batch shape: torch.Size([64, 3, 96, 96]), Labels shape: torch.Size([64])
Value range: [-1.00, 1.00]
Number of classes: 10


In [6]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super(SimpleCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 12 * 12, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )
        
    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)


In [9]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        
        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item() * x.size(0)
        _, predicted = torch.max(logits, 1)
        total += y.size(0)
        correct += (predicted == y).sum().item()
        
    return total_loss / total, 100 * correct / total

def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            loss = criterion(logits, y)
            
            total_loss += loss.item() * x.size(0)
            _, predicted = torch.max(logits, 1)
            total += y.size(0)
            correct += (predicted == y).sum().item()
            
    return total_loss / total, 100 * correct / total

results = []

def run_classification_experiment(exp_id, model, train_loader, val_loader, epochs, lr=1e-3, model_name="SimpleCNN"):
    model = model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    best_val_acc = -1
    best_state = None
    
    print(f"\n--- Running {exp_id} ---")
    
    for epoch in range(epochs):
        t_loss, t_acc = train_one_epoch(model, train_loader, optimizer, criterion, device)
        v_loss, v_acc = evaluate(model, val_loader, criterion, device)
        
        history['train_loss'].append(t_loss)
        history['val_loss'].append(v_loss)
        history['train_acc'].append(t_acc)
        history['val_acc'].append(v_acc)
        
        if v_acc > best_val_acc:
            best_val_acc = v_acc
            best_state = copy.deepcopy(model.state_dict())
        
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"Epoch {epoch+1}: Train Loss {t_loss:.4f}, Val Acc {v_acc:.2f}%")
    
    results.append({
        'experiment_id': exp_id,
        'task': 'classification',
        'dataset': 'STL10',
        'seed': SEED,
        'model_summary': model_name,
        'optimizer': 'Adam',
        'lr': lr,
        'epochs_trained': epochs,
        'best_val_accuracy': best_val_acc,
        'test_accuracy': None,
        'precision': None,
        'recall': None,
        'mean_iou': None,
        'notes': ''
    })
    
    return history, best_state, best_val_acc


In [10]:
cnn_base = SimpleCNN(num_classes=NUM_CLASSES_A)
h_c1, state_c1, acc_c1 = run_classification_experiment('C1', cnn_base, train_loader_base, val_loader_base, NUM_EPOCHS, model_name="SimpleCNN-Base")

cnn_aug = SimpleCNN(num_classes=NUM_CLASSES_A)
h_c2, state_c2, acc_c2 = run_classification_experiment('C2', cnn_aug, train_loader_aug, val_loader_aug, NUM_EPOCHS, model_name="SimpleCNN-Aug")

resnet18 = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
num_features = resnet18.fc.in_features
resnet18.fc = nn.Linear(num_features, NUM_CLASSES_A)

for param in resnet18.parameters():
    param.requires_grad = False
for param in resnet18.fc.parameters():
    param.requires_grad = True

train_dataset_resnet = torchvision.datasets.STL10(root='./data', split='train', download=True, transform=transform_resnet)
val_dataset_resnet = torchvision.datasets.STL10(root='./data', split='train', download=True, transform=transform_resnet)
train_size_r = int(0.8 * len(train_dataset_resnet))
val_size_r = len(train_dataset_resnet) - train_size_r
train_dataset_resnet, val_dataset_resnet = random_split(train_dataset_resnet, [train_size_r, val_size_r], generator=torch.Generator().manual_seed(SEED))

train_loader_resnet = DataLoader(train_dataset_resnet, batch_size=BATCH_SIZE, shuffle=True)
val_loader_resnet = DataLoader(val_dataset_resnet, batch_size=BATCH_SIZE, shuffle=False)

h_c3, state_c3, acc_c3 = run_classification_experiment('C3', resnet18, train_loader_resnet, val_loader_resnet, NUM_EPOCHS, lr=1e-3, model_name="ResNet18-HeadOnly")

resnet18_finetune = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
num_features = resnet18_finetune.fc.in_features
resnet18_finetune.fc = nn.Linear(num_features, NUM_CLASSES_A)

for param in resnet18_finetune.parameters():
    param.requires_grad = False
for param in resnet18_finetune.layer4.parameters():
    param.requires_grad = True
for param in resnet18_finetune.fc.parameters():
    param.requires_grad = True

h_c4, state_c4, acc_c4 = run_classification_experiment('C4', resnet18_finetune, train_loader_resnet, val_loader_resnet, NUM_EPOCHS, lr=1e-4, model_name="ResNet18-FineTune")

best_exp = max(results[:4], key=lambda x: x['best_val_accuracy'])
best_exp_id = best_exp['experiment_id']
print(f"\nBest classification experiment: {best_exp_id} with val_accuracy: {best_exp['best_val_accuracy']:.2f}%")

if best_exp_id == 'C1':
    best_state = state_c1
    best_loader = test_loader
    best_transform = transform_base
elif best_exp_id == 'C2':
    best_state = state_c2
    best_loader = test_loader
    best_transform = transform_base
else:
    best_state = state_c3 if best_exp_id == 'C3' else state_c4
    test_dataset_resnet = torchvision.datasets.STL10(root='./data', split='test', download=True, transform=transform_resnet)
    best_loader = DataLoader(test_dataset_resnet, batch_size=BATCH_SIZE, shuffle=False)
    best_transform = transform_resnet

best_model = SimpleCNN(num_classes=NUM_CLASSES_A) if best_exp_id in ['C1', 'C2'] else models.resnet18(weights=None)
if best_exp_id not in ['C1', 'C2']:
    best_model.fc = nn.Linear(best_model.fc.in_features, NUM_CLASSES_A)

best_model.load_state_dict(best_state)
best_model = best_model.to(device)

criterion = nn.CrossEntropyLoss()
test_loss, test_acc = evaluate(best_model, best_loader, criterion, device)

for r in results:
    if r['experiment_id'] == best_exp_id:
        r['test_accuracy'] = round(test_acc, 2)

print(f"\n=== FINAL TEST ACCURACY (Best Model {best_exp_id}): {test_acc:.2f}% ===")



--- Running C1 ---
Epoch 1: Train Loss 2.5408, Val Acc 33.50%
Epoch 5: Train Loss 1.4956, Val Acc 47.30%
Epoch 10: Train Loss 1.2624, Val Acc 55.60%
Epoch 15: Train Loss 1.0628, Val Acc 54.80%

--- Running C2 ---
Epoch 1: Train Loss 2.4114, Val Acc 27.80%
Epoch 5: Train Loss 1.6529, Val Acc 44.40%
Epoch 10: Train Loss 1.5375, Val Acc 48.50%
Epoch 15: Train Loss 1.4643, Val Acc 50.00%
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /Users/vladislava/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████████████████████████████████| 44.7M/44.7M [00:05<00:00, 9.36MB/s]



--- Running C3 ---
Epoch 1: Train Loss 1.0472, Val Acc 91.70%
Epoch 5: Train Loss 0.1857, Val Acc 94.60%
Epoch 10: Train Loss 0.1229, Val Acc 94.10%
Epoch 15: Train Loss 0.0918, Val Acc 94.70%

--- Running C4 ---
Epoch 1: Train Loss 0.6076, Val Acc 93.70%
Epoch 5: Train Loss 0.0110, Val Acc 94.70%
Epoch 10: Train Loss 0.0031, Val Acc 95.00%
Epoch 15: Train Loss 0.0015, Val Acc 94.80%

Best classification experiment: C4 with val_accuracy: 95.30%

=== FINAL TEST ACCURACY (Best Model C4): 95.40% ===


In [13]:
torch.save(best_state, "artifacts/best_classifier.pt")

best_config_data = {
    "dataset": "STL10",
    "seed": SEED,
    "model_architecture": best_exp['model_summary'],
    "num_classes": NUM_CLASSES_A,
    "optimizer": "Adam",
    "lr": best_exp['lr'],
    "batch_size": BATCH_SIZE,
    "epochs_trained": best_exp['epochs_trained'],
    "transforms": "base" if best_exp_id in ['C1', 'C2'] else "resnet",
    "best_val_accuracy": round(best_exp['best_val_accuracy'], 2),
    "test_accuracy": round(test_acc, 2)
}

with open("artifacts/best_classifier_config.json", "w", encoding='utf-8') as f:
    json.dump(best_config_data, f, indent=2, ensure_ascii=False)

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
if best_exp_id == 'C1':
    h_best = h_c1
elif best_exp_id == 'C2':
    h_best = h_c2
elif best_exp_id == 'C3':
    h_best = h_c3
else:
    h_best = h_c4

plt.plot(h_best['train_loss'], label='Train Loss')
plt.plot(h_best['val_loss'], label='Val Loss')
plt.title(f"{best_exp_id} Loss (Best Model)")
plt.xlabel("Epoch")
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(h_best['train_acc'], label='Train Acc')
plt.plot(h_best['val_acc'], label='Val Acc')
plt.title(f"{best_exp_id} Accuracy (Best: {best_exp['best_val_accuracy']:.2f}%)")
plt.xlabel("Epoch")
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.savefig("artifacts/figures/classification_curves_best.png")
plt.close()

plt.figure(figsize=(10, 6))
exp_ids = ['C1', 'C2', 'C3', 'C4']
val_accs = [r['best_val_accuracy'] for r in results[:4]]
colors = ['blue', 'green', 'orange', 'red']
plt.bar(exp_ids, val_accs, color=colors)
plt.title('Classification Experiments Comparison (C1-C4)')
plt.xlabel('Experiment')
plt.ylabel('Best Val Accuracy (%)')
plt.ylim(0, 100)
for i, v in enumerate(val_accs):
    plt.text(i, v + 2, f'{v:.1f}%', ha='center')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig("artifacts/figures/classification_compare.png")
plt.close()

transform_pil = transforms.Compose([
    transforms.Resize((96, 96))
])

train_dataset_pil = torchvision.datasets.STL10(root='./data', split='train', download=True, transform=transform_pil)

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
sample_idx = np.random.randint(0, len(train_dataset_pil))

original = train_dataset_pil[sample_idx][0]
aug1 = transform_aug(original)
aug2 = transform_aug(original)
aug3 = transform_aug(original)

axes[0, 0].imshow(original)
axes[0, 0].set_title('Original')
axes[0, 0].axis('off')

axes[0, 1].imshow(aug1.permute(1, 2, 0))
axes[0, 1].set_title('Augmentation 1')
axes[0, 1].axis('off')

axes[0, 2].imshow(aug2.permute(1, 2, 0))
axes[0, 2].set_title('Augmentation 2')
axes[0, 2].axis('off')

axes[1, 0].imshow(aug3.permute(1, 2, 0))
axes[1, 0].set_title('Augmentation 3')
axes[1, 0].axis('off')

axes[1, 1].axis('off')
axes[1, 2].axis('off')

plt.tight_layout()
plt.savefig("artifacts/figures/augmentations_preview.png")
plt.close()

seg_dataset = torchvision.datasets.OxfordIIITPet(root='./data', split='trainval', target_types='segmentation', download=True)

def get_sample_indices(dataset, n=5):
    indices = []
    for i in range(len(dataset)):
        try:
            img, mask = dataset[i]
            if mask is not None:
                indices.append(i)
                if len(indices) >= n:
                    break
        except:
            continue
    return indices

sample_indices = get_sample_indices(seg_dataset, 5)

seg_model = torchvision.models.segmentation.deeplabv3_resnet50(weights=torchvision.models.segmentation.DeepLabV3_ResNet50_Weights.COCO_WITH_VOC_LABELS_V1)
seg_model = seg_model.to(device)
seg_model.eval()

def process_segmentation(model, img_tensor, device, threshold_mode='default'):
    model.eval()
    with torch.no_grad():
        input_tensor = img_tensor.unsqueeze(0).to(device)
        output = model(input_tensor)['out'][0]
        output = output.cpu().numpy()
        
        if threshold_mode == 'default':
            pred_mask = output.argmax(0)
        else:
            foreground_class = 15
            pred_mask = (output[foreground_class] > 0.5).astype(np.int64)
        
        return pred_mask

def calculate_mean_iou(pred_mask, gt_mask, num_classes=2):
    ious = []
    for cls in range(num_classes):
        pred_inds = pred_mask == cls
        gt_inds = gt_mask == cls
        intersection = np.logical_and(pred_inds, gt_inds).sum()
        union = np.logical_or(pred_inds, gt_inds).sum()
        if union > 0:
            ious.append(intersection / union)
    return np.mean(ious) if ious else 0.0

v1_ious = []
v2_ious = []

fig, axes = plt.subplots(5, 4, figsize=(20, 25))

for idx, sample_idx in enumerate(sample_indices):
    img, mask = seg_dataset[sample_idx]
    img_tensor = transforms.ToTensor()(img)
    
    pred_v1 = process_segmentation(seg_model, img_tensor, device, threshold_mode='default')
    pred_v2 = process_segmentation(seg_model, img_tensor, device, threshold_mode='alternative')
    
    mask_np = np.array(mask)
    mask_np_binary = (mask_np > 0).astype(np.int64)
    pred_v1_binary = (pred_v1 > 0).astype(np.int64)
    pred_v2_binary = (pred_v2 > 0).astype(np.int64)
    
    iou_v1 = calculate_mean_iou(pred_v1_binary, mask_np_binary, num_classes=2)
    iou_v2 = calculate_mean_iou(pred_v2_binary, mask_np_binary, num_classes=2)
    
    v1_ious.append(iou_v1)
    v2_ious.append(iou_v2)
    
    axes[idx, 0].imshow(img)
    axes[idx, 0].set_title(f'Original Image {idx+1}')
    axes[idx, 0].axis('off')
    
    axes[idx, 1].imshow(mask_np, cmap='tab20')
    axes[idx, 1].set_title('Ground Truth Mask')
    axes[idx, 1].axis('off')
    
    axes[idx, 2].imshow(pred_v1_binary, cmap='tab20')
    axes[idx, 2].set_title(f'V1 Prediction (IoU={iou_v1:.3f})')
    axes[idx, 2].axis('off')
    
    axes[idx, 3].imshow(pred_v2_binary, cmap='tab20')
    axes[idx, 3].set_title(f'V2 Prediction (IoU={iou_v2:.3f})')
    axes[idx, 3].axis('off')

plt.tight_layout()
plt.savefig("artifacts/figures/segmentation_examples.png")
plt.close()

mean_iou_v1 = np.mean(v1_ious)
mean_iou_v2 = np.mean(v2_ious)

results.append({
    'experiment_id': 'V1',
    'task': 'segmentation',
    'dataset': 'OxfordIIITPet',
    'seed': SEED,
    'model_summary': 'DeepLabV3-ResNet50-Default',
    'optimizer': '',
    'lr': 0,
    'epochs_trained': 0,
    'best_val_accuracy': None,
    'test_accuracy': None,
    'precision': None,
    'recall': None,
    'mean_iou': round(mean_iou_v1, 4),
    'notes': 'Default argmax postprocessing'
})

results.append({
    'experiment_id': 'V2',
    'task': 'segmentation',
    'dataset': 'OxfordIIITPet',
    'seed': SEED,
    'model_summary': 'DeepLabV3-ResNet50-Threshold',
    'optimizer': '',
    'lr': 0,
    'epochs_trained': 0,
    'best_val_accuracy': None,
    'test_accuracy': None,
    'precision': None,
    'recall': None,
    'mean_iou': round(mean_iou_v2, 4),
    'notes': 'Alternative threshold postprocessing'
})

plt.figure(figsize=(10, 6))
exp_ids_seg = ['V1', 'V2']
iou_vals = [mean_iou_v1, mean_iou_v2]
colors_seg = ['purple', 'brown']
plt.bar(exp_ids_seg, iou_vals, color=colors_seg)
plt.title('Segmentation Modes Comparison (V1 vs V2)')
plt.xlabel('Mode')
plt.ylabel('Mean IoU')
plt.ylim(0, 1)
for i, v in enumerate(iou_vals):
    plt.text(i, v + 0.02, f'{v:.3f}', ha='center')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig("artifacts/figures/segmentation_metrics.png")
plt.close()

df_results = pd.DataFrame(results)
df_results.to_csv("artifacts/runs.csv", index=False)
print("Saved artifacts/runs.csv")

print(f"\n=== SEGMENTATION RESULTS ===")
print(f"V1 (Default) Mean IoU: {mean_iou_v1:.4f}")
print(f"V2 (Alternative) Mean IoU: {mean_iou_v2:.4f}")

print(f"Python: {sys.version.split()[0]}")
print(f"torch: {torch.__version__}")
print(f"torchvision: {torchvision.__version__}")
print(f"Device: {'CUDA' if torch.cuda.is_available() else 'CPU'}")

Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [-1.0..0.9607843].
Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [-0.99215686..0.70980394].
Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [-0.8352941..0.827451].
100%|████████████████████████████████████████| 792M/792M [02:23<00:00, 5.53MB/s]
100%|██████████████████████████████████████| 19.2M/19.2M [00:02<00:00, 6.52MB/s]


Downloading: "https://download.pytorch.org/models/deeplabv3_resnet50_coco-cd0a2569.pth" to /Users/vladislava/.cache/torch/hub/checkpoints/deeplabv3_resnet50_coco-cd0a2569.pth


100%|████████████████████████████████████████| 161M/161M [00:14<00:00, 11.3MB/s]


Saved artifacts/runs.csv

=== SEGMENTATION RESULTS ===
V1 (Default) Mean IoU: 0.1670
V2 (Alternative) Mean IoU: 0.6262
Python: 3.12.12
torch: 2.10.0
torchvision: 0.25.0
Device: CPU
